In [ ]:
class XYZAwareLearnedSessionAdapter(nn.Module):
    """Coordinate-aware spike embeddings + per-task heads.

    - Shared MLPs (coord_mlp_in / coord_mlp_out) dynamically generate 
      w_in (N, d) and w_out (N, d) from 3D unit coordinates.
    - Per-session b_out (N,) bias initialized to mean firing rate.
    - Per-task embedding (d,) to condition the model on the current task.
    """
    def __init__(self, n_units_per_session, task_specs, d, device,
                 mean_rates=None, session_coords=None):
        super().__init__()
        self.d = d
        self.session_coords = [
            torch.tensor(c, dtype=torch.float32, device=device)
            for c in session_coords
        ]
        
        from gradient_normalizer import GradientNormalizer
        self.grad_normalizers = nn.ModuleDict({
            spec.name: GradientNormalizer(normalizer_shape=[1], scale=1.0)
            for spec in task_specs
        })
        self.task_embeds = nn.ParameterDict({
            spec.name: nn.Parameter(torch.randn(d, device=device) * 0.02)
            for spec in task_specs
        })
        
        # coordnate aware mlp
        self.coord_mlp_in = nn.Sequential(
            nn.Linear(3, 64),
            nn.GELU(),
            nn.Linear(64, d),
        )
        
        self.coord_mlp_out = nn.Sequential(
            nn.Linear(3, 64),
            nn.GELU(),
            nn.Linear(64, d),
        )

        # b_out initialized so that relu(0 + b_out) = mean firing rate per neuron
        self._b_out = nn.ParameterList([
            nn.Parameter(torch.from_numpy(mean_rates[i]).float().to(device))
            for i in range(len(n_units_per_session))
        ])
        # Per-task prediction heads
        task_heads = {}
        for spec in task_specs:
            if isinstance(spec, RegressionTask):
                task_heads[spec.name] = nn.Parameter(
                    torch.randn(spec.d_out, d, device=device) * (1.0 / d**0.5))
            elif isinstance(spec, ClassificationTask):
                task_heads[spec.name] = nn.Parameter(
                    torch.randn(spec.n_classes, d, device=device) * (1.0 / d**0.5))
        self.task_heads = nn.ParameterDict(task_heads)

    def w_in(self, session, visible_idx=None):
        coords = self.session_coords[session]
        if visible_idx is not None:
            coords = coords[visible_idx]
        return self.coord_mlp_in(coords)

    def w_out(self, session):
        return self.coord_mlp_out(self.session_coords[session])

    def b_out(self, session):
        return self._b_out[session]